<a href="https://colab.research.google.com/github/reganq/csc311-project/blob/main/random_forest_model_family.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decision Tree / Random Forest Model Tree


## Import Data, libraries, define constants, etc.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
import statistics as stats

url = "https://raw.githubusercontent.com/reganq/csc311-project/refs/heads/main/cleaned_data.csv"
df = pd.read_csv(url)
# features : array of features in df
features = df.columns.tolist()
features.remove("painting")
# separate features for training from validation
#TBD
NUM_TREES = 100


C:\Users\ecorb\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Function for decision tree

In [2]:
#define function to make tree, called decision tree, from a dataframe X
def decision_tree(df=df, criterion="entropy", max_depth=3, min_samples_leaf=1, max_features="sqrt"):
  """
  run decision tree model on dataframe X
  criterion=criterion, max_depth=max_depth, min_samples_leaf=min_samples_leaf, max_features=max_features
  return the decision tree model that is fit as per the parameters in the function.
  """
  X=df[features]
  t=df["painting"]
  tree = DecisionTreeClassifier(criterion=criterion, max_depth=max_depth, min_samples_leaf=min_samples_leaf, max_features=max_features)
  tree.fit(X, t)
  return tree

## Function for random forest



In [3]:
def construct_forest(df=df, criterion="entropy", max_depth=3, min_samples_leaf=1, max_features="sqrt", boot_size=100):
  """
  construct random forest, <forest>, in the form of a list of decision trees.
  fit each tree on a unique bootstrapped sample of <df> with replacement and size <boot_size>.
  return <forest>.
  """
  forest = []
  for t in range(NUM_TREES):
    np.random.seed(311)
    df_boot = df.sample(n=boot_size, replace=True)
    forest.append(decision_tree(df=df_boot, criterion=criterion, max_depth=max_depth, min_samples_leaf=min_samples_leaf, max_features=max_features))
  return forest

## Hypeparameter training


## Brenden fooling around

In [54]:
forest = construct_forest(max_depth=3)
votes = []
for tree in forest:
  votes.append(tree.predict(df[features])[120])
print(votes)
print(stats.mode(votes))

['The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night', 'The Starry Night',

In [55]:
from sklearn.tree import export_text
text_representation = export_text(forest[2], feature_names=features)
print(text_representation)
tree = forest[5]
left_children = tree.tree_.children_left
right_children = tree.tree_.children_right
feature_splits = tree.tree_.feature # array of the feature index used for splitting
thresholds = tree.tree_.threshold # array of the split points/thresholds

|--- text_starry_music <= 0.01
|   |--- evokes_sombre <= 3.50
|   |   |--- text_clock_feels <= 0.02
|   |   |   |--- class: The Water Lily Pond
|   |   |--- text_clock_feels >  0.02
|   |   |   |--- class: The Water Lily Pond
|   |--- evokes_sombre >  3.50
|   |   |--- evokes_content <= 3.50
|   |   |   |--- class: The Persistence of Memory
|   |   |--- evokes_content >  3.50
|   |   |   |--- class: The Water Lily Pond
|--- text_starry_music >  0.01
|   |--- unique_id <= 393.50
|   |   |--- prominent_colour_count <= 1.50
|   |   |   |--- class: The Water Lily Pond
|   |   |--- prominent_colour_count >  1.50
|   |   |   |--- class: The Starry Night
|   |--- unique_id >  393.50
|   |   |--- class: The Persistence of Memory



In [56]:
feature_splits

array([27,  3, 22, -2, -2,  4, -2, -2,  0,  7, -2, -2, -2], dtype=int64)

In [57]:
thresholds

array([ 1.19152693e-02,  3.50000000e+00,  1.56749841e-02, -2.00000000e+00,
       -2.00000000e+00,  3.50000000e+00, -2.00000000e+00, -2.00000000e+00,
        3.93500000e+02,  1.50000000e+00, -2.00000000e+00, -2.00000000e+00,
       -2.00000000e+00])

In [58]:
left_children

array([ 1,  2,  3, -1, -1,  6, -1, -1,  9, 10, -1, -1, -1], dtype=int64)